https://medium.com/@kbouziane.ai/harnessing-rag-for-text-tables-and-images-a-comprehensive-guide-ca4d2d420219

In [1]:
%%capture
!pip install "langchain>=0.0.331rc2" "unstructured[all-docs]" langchain_community unstructured==0.12.5 pillow pydantic lxml pillow matplotlib chromadb tiktoken watermark


In [2]:
%%capture
!apt-get update


!sudo apt-get install poppler-utils tesseract-ocr

In [3]:
%%capture
!pip install google-generativeai langchain_google_genai huggingface_hub

In [4]:
# This cell is ruuning because partition_pdf is througing error while parsing PDF.
!pip install unstructured==0.12.5 unstructured-inference==0.7.24 #unstructured_pytesseract

ERROR: Invalid requirement: '#unstructured_pytesseract'


In [5]:
# import IPython

# IPython.Application.instance().kernel.do_shutdown(True)

In [6]:
import os
import io
import uuid
import base64
from base64 import b64decode
import numpy as np
from PIL import Image, UnidentifiedImageError

from unstructured.partition.pdf import partition_pdf

from langchain.chat_models import ChatOpenAI
from langchain.schema.messages import HumanMessage, SystemMessage
from langchain.vectorstores import Chroma
from langchain.storage import InMemoryStore
from langchain.schema.document import Document
from langchain.embeddings import OpenAIEmbeddings
from langchain.retrievers.multi_vector import MultiVectorRetriever
from langchain.chat_models import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain.schema.output_parser import StrOutputParser
from langchain.schema.runnable import RunnablePassthrough, RunnableLambda

from operator import itemgetter

ImportError: cannot import name 'etree' from 'lxml' (c:\Users\riman\VSCode\GenAI_Projects\GenAI\gen_env\lib\site-packages\lxml\__init__.py)

In [ ]:
# os.environ["OPENAI_API_KEY"] = ''
# openai.api_key = os.environ["OPENAI_API_KEY"]



# New Section

In [ ]:
# load the pdf file to drive
# split the file to text, table and images
def doc_partition(path,file_name):
  raw_pdf_elements = partition_pdf(
    filename=path + file_name,
    image_output_dir_path=path,
     extract_images_in_pdf=True,
    infer_table_structure=True,
    chunking_strategy="by_title", # hi_res
    max_characters=4000,
    new_after_n_chars=3800,
    combine_text_under_n_chars=2000,
    )

  return raw_pdf_elements
poppler_path = "/content/"
path = "/content/"
file_name = "IF10244.pdf"

raw_pdf_elements = doc_partition(path,file_name)

config.json:   0%|          | 0.00/1.47k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/115M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/46.8M [00:00<?, ?B/s]

Some weights of the model checkpoint at microsoft/table-transformer-structure-recognition were not used when initializing TableTransformerForObjectDetection: ['model.backbone.conv_encoder.model.layer2.0.downsample.1.num_batches_tracked', 'model.backbone.conv_encoder.model.layer3.0.downsample.1.num_batches_tracked', 'model.backbone.conv_encoder.model.layer4.0.downsample.1.num_batches_tracked']
- This IS expected if you are initializing TableTransformerForObjectDetection from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TableTransformerForObjectDetection from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [ ]:
raw_pdf_elements[2].metadata.to_dict()

In [ ]:
# appending texts and tables from the pdf file
def data_category(raw_pdf_elements): # we may use decorator here
    tables = []
    texts = []
    for element in raw_pdf_elements:
        if "unstructured.documents.elements.Table" in str(type(element)):
           tables.append(str(element))
        elif "unstructured.documents.elements.CompositeElement" in str(type(element)):
           texts.append(str(element))
    data_category = [texts,tables]
    return data_category
texts = data_category(raw_pdf_elements)[0]
tables = data_category(raw_pdf_elements)[1]

In [ ]:
[element.metadata.page_number for element in raw_pdf_elements if "unstructured.documents.elements.CompositeElement" in str(type(element))]

In [ ]:
# from huggingface_hub import login
# from langchain_community.llms import HuggingFaceHub

# from google.colab import userdata
# HF_Token = userdata.get('HUGGINGFACEHUB_API_TOKEN')

# os.environ['HUGGINGFACEHUB_API_TOKEN']=HF_Token

# login(token=HF_Token)

# repo_id = "google/gemma-2-2b-it" #"google/flan-t5-xl" # Replace with your desired model
# llm = HuggingFaceHub(repo_id=repo_id, model_kwargs={"temperature":0.5, "max_length":64})



In [2]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
import google.generativeai as genai
from langchain_google_genai import ChatGoogleGenerativeAI

# from google.colab import userdata
# HF_Token = userdata.get('GOOGLE_API_KEY')

# os.environ['GOOGLE_API_KEY']=HF_Token

genai.configure(api_key=os.environ['GOOGLE_API_KEY'])

NameError: name 'userdata' is not defined

In [ ]:
tables

In [ ]:
# function to take tables and text and then summarize tables only,
def tables_summarize(data_category):
    prompt_text = """You are an assistant tasked with summarizing tables and text. \
                    Give a concise summary of the table or text. Table or text chunk: {element} """

    prompt = ChatPromptTemplate.from_template(prompt_text)
    # model = ChatOpenAI(temperature=0, model="gpt-3.5-turbo-instruct")
    model = ChatGoogleGenerativeAI(model="gemini-pro", temperature=0.3)
    summarize_chain = {"element": lambda x: x} | prompt | model | StrOutputParser()
    table_summaries = summarize_chain.batch(tables, {"max_concurrency": 5})
    #text_summaries =  summarize_chain.batch(data_category[0], {"max_concurrency": 5})# no need to summarize

    return table_summaries
table_summaries = tables_summarize(data_category)
text_summaries = texts

In [ ]:
table_summaries

In [ ]:
def encode_image(image_path):
    ''' Getting the base64 string '''
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

def image_captioning(img_base64,prompt):
    ''' Image summary '''
    # chat = ChatOpenAI(model="gpt-4-vision-preview",
    #                   max_tokens=1024)

    chat = ChatGoogleGenerativeAI(model="gemini-1.5-flash", temperature=0.1)

    msg = chat.invoke(
        [
            HumanMessage(
                content=[
                    {"type": "text", "text":prompt},
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/jpeg;base64,{img_base64}"
                        },
                    },
                ]
            )
        ]
    )
    return msg.content

In [ ]:
for m in genai.list_models():
  if 'generateContent' in m.supported_generation_methods:
    print(m.name)

In [ ]:
import time
# Store base64 encoded images
img_base64_list = []

# Store image summaries
image_summaries = []

# Prompt
prompt = "Describe the image in detail. Be specific about graphs, such as bar plots."
path1 = path + '/figures'
# Read images, encode to base64 strings
for img_file in sorted(os.listdir(path1)):
    if img_file.endswith('.jpg'):
        img_path = os.path.join(path1, img_file)
        print(img_path)
        base64_image = encode_image(img_path)
        img_base64_list.append(base64_image)
        #image_summaries.append(image_captioning(base64_image,prompt))
        img_cap = image_captioning(base64_image,prompt)
        # time.sleep(10)
        image_summaries.append(img_cap)


In [ ]:
image_summaries

In [ ]:
img_base64_list

In [ ]:
def split_image_text_types(docs):
    ''' Split base64-encoded images and texts '''
    b64 = []
    text = []
    for doc in docs:
        try:
            b64decode(doc)
            b64.append(doc)
        except Exception as e:
            text.append(doc)
    return {
        "images": b64,
        "texts": text
    }

In [ ]:
# Create a vectore base and store text,table,images and their index

In [ ]:
# Add raw docs and doc summaries to Multi Vector Retriever.
# The vectorstore to use to index the child chunks
# vectorstore = Chroma(collection_name="multi_modal_rag",
#                      embedding_function=OpenAIEmbeddings())

vectorstore = Chroma(collection_name="multi_modal_rag",
                     embedding_function=GoogleGenerativeAIEmbeddings(model = "models/embedding-001"))

# The storage layer for the parent documents
store = InMemoryStore()
id_key = "doc_id"

# The retriever (empty to start)
retriever = MultiVectorRetriever(
    vectorstore=vectorstore,
    docstore=store,
    id_key=id_key,
)

# Add texts
doc_ids = [str(uuid.uuid4()) for _ in texts]

summary_texts = [
    Document(
        page_content=s,
        metadata={
            id_key: doc_ids[i],
            'page_number': next(
                (element.metadata.page_number for element in raw_pdf_elements
                 if "unstructured.documents.elements.CompositeElement" in str(type(element))
                 and element.metadata.page_number is not None),
                None
            )
        }
    )
    for i, s in enumerate(text_summaries)
]

# summary_texts = [
#     Document(page_content=s, metadata={id_key: doc_ids[i]})
#     for i, s in enumerate(text_summaries)
# ]

retriever.vectorstore.add_documents(summary_texts)
retriever.docstore.mset(list(zip(doc_ids, texts)))

# Add tables
table_ids = [str(uuid.uuid4()) for _ in tables]
summary_tables = [
    Document(page_content=s, metadata={id_key: table_ids[i]})
    for i, s in enumerate(table_summaries)
]
retriever.vectorstore.add_documents(summary_tables)
retriever.docstore.mset(list(zip(table_ids, tables)))

# Add image summaries
img_ids = [str(uuid.uuid4()) for _ in img_base64_list]
summary_img = [
    Document(page_content=s, metadata={id_key: img_ids[i]})
    for i, s in enumerate(image_summaries)
]
retriever.vectorstore.add_documents(summary_img)
retriever.docstore.mset(list(zip(img_ids, img_base64_list)))

In [ ]:
from operator import itemgetter
from langchain.schema.runnable import RunnablePassthrough, RunnableLambda

def prompt_func(dict):
    format_texts = "\n".join(dict["context"]["texts"])
    return [
        HumanMessage(
            content=[
                {"type": "text", "text": f"""Answer the question based only on the following context, which can include text, tables, and the below image:
Question: {dict["question"]}

Text and tables:
{format_texts}
"""},
                {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{dict['context']['images'][0]}"}},
            ]
        )
    ]

# model = ChatOpenAI(temperature=0, model="gpt-4-vision-preview", max_tokens=1024)
model = ChatGoogleGenerativeAI(model="gemini-1.5-flash", temperature=0.1)


# RAG pipeline
chain = (
    {"context": retriever | RunnableLambda(split_image_text_types), "question": RunnablePassthrough()}
    | RunnableLambda(prompt_func)
    | model
    | StrOutputParser()
)

In [ ]:
chain.invoke(
    "What is the change in wild fires from 1993 to 2022 include chart?"
)